# 아마존 PDP 데이터와 매출의 연관성 분석
### - XGBoost 

In [82]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor
#import ace_tools as tools

In [84]:
df = pd.read_csv('amz_pdp_price_sales_daily_0613_top10.csv')

In [86]:
# 1) 전처리
df = df.dropna(subset=['crawl_date','asin','rating','ratings_cnt','salesrank2','bw_price','units'])
df['date'] = pd.to_datetime(df['crawl_date'])
df = df.sort_values(['asin','date'])

In [88]:
print(df.dtypes)

crawl_date             object
asin                   object
rating                float64
ratings_cnt           float64
salesrank1            float64
salesrank2            float64
salesrank3            float64
bw_price              float64
revenue               float64
units                 float64
date           datetime64[ns]
dtype: object


In [90]:
# 2) Lag 피처 생성
grouped = df.groupby('asin')
df['units_lag1'] = grouped['units'].shift(1)
#df['units_lag30'] = grouped['units'].shift(30)
#df = df.dropna(subset=['units_lag7','units_lag30'])

In [92]:
# 모델 성능 저장용 리스트
results = []
features = ['rating', 'ratings_cnt', 'salesrank2', 'bw_price', 'units_lag1']
#features = ['rating', 'ratings_cnt', 'salesrank2', 'bw_price', 'units_lag7', 'units_lag30']

In [94]:
# ASIN별 모델링 수행
for asin, sub in df.groupby('asin'):
    sub = sub.dropna(subset=features + ['units'])
    if len(sub) < 10:  # 샘플 수 부족 시 건너뛰기
        continue

    # 학습/테스트 분할 (80% / 20%)
    split_idx = int(len(sub) * 0.8)
    train = sub.iloc[:split_idx]
    test  = sub.iloc[split_idx:]

    X_train, y_train = train[features], train['units']
    X_test,  y_test  = test[features],  test['units']

    # XGBoost 모델 학습
    model = XGBRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)

    # 예측 및 성능 평가
    y_pred = model.predict(X_test)
    r2   = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    results.append({
        'asin': asin,
        'R2': r2,
        'RMSE': rmse
    })

In [96]:
# 결과를 DataFrame으로 정리하여 출력
metrics_df = pd.DataFrame(results)
print(metrics_df)

         asin         R2        RMSE
0  B0CKYRS739   0.628608   87.043536
1  B0CKYY27YP   0.309176  164.817352
2  B0CKYYHD47  -0.138514   17.288438
3  B0CKYZ3B83  -0.175431  168.925363
4  B0CKYZ4DJB   0.371555  104.017164
5  B0CKYZCVXK   0.590486  186.550408
6  B0CKYZPPMJ   0.270052   37.270528
7  B0CKZ1CK1H  -0.867171   52.276809
8  B0CP1LR1PW -14.569008  113.049028
9  B0CSJTKX7K   0.023213   50.763109
